# Scale-invariant Barcelona models — training

The ablation study showed Barcelona's noise signal lives almost entirely in the 10 **network** features
(6 `dist_to_*` + 4 centralities), but those are exactly the features that go out-of-distribution when the
model is transferred to another city — their raw values depend on network size / urban fabric (closeness
shrinks as a network grows, `dist_to_trunk` explodes in big cities). This notebook makes those features
**scale-invariant** and trains two variants:

- **SI-10** — only the 10 network features, percentile-ranked.
- **SI-18** — **all 18 features**, with the 10 network features percentile-ranked and the 8 **local**
  features (`road_category, width, signals, transport, pois, green, industrial, commercial`) left raw.
  The local features are already roughly city-comparable (a road-class code, width in metres, point
  densities, land-use percentages), so only the network features need normalizing.

## How the scale-invariant features are built

For each feature `f` in the 10 network features, **within the city**:

```
df[f + '_pct'] = df[f].rank(pct=True)     # empirical CDF: fraction of the city's streets with value <= this one
```

The percentile is dimensionless and city-relative: "this street is in the 90th percentile of closeness
*for its city*" means the same thing everywhere regardless of size or segment count. Because the rank
transform is **monotonic**, it does not change the tree models *within* Barcelona — SI-18's RF/XGBoost
held-out scores should equal the original full-18 model (RF 0.749 acc / 0.704 R²); SI-10 should equal the
ablation's raw `only-distances+centrality` (RF 0.751 / 0.696). The payoff is in transfer
(`02_test_all_cities_scale_invariant.ipynb`).

## Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score)
import xgboost as xgb

## Features and the percentile transform

In [2]:
NETWORK_10 = [
    'dist_to_trunk', 'dist_to_primary', 'dist_to_secondary',
    'dist_to_tertiary', 'dist_to_residential', 'dist_to_living_street',
    'betweenness', 'closeness_global', 'closeness_400', 'straightness',
]
LOCAL_8 = ['road_category', 'width', 'signals', 'transport', 'pois',
           'green', 'industrial', 'commercial']
PCT_COLS = [f + '_pct' for f in NETWORK_10]

SI10_FEATS = PCT_COLS                 # 10 network features, percentile-ranked
SI18_FEATS = LOCAL_8 + PCT_COLS       # all 18, network features percentile-ranked

def add_pct(df):
    """Add per-city percentile-rank versions of the 10 network features (scale-invariant)."""
    out = df.copy()
    for f in NETWORK_10:
        out[f + '_pct'] = out[f].rank(pct=True)
    return out

print('SI-10:', len(SI10_FEATS), 'features |  SI-18:', len(SI18_FEATS), 'features')

SI-10: 10 features |  SI-18: 18 features


## Load Barcelona and build the scale-invariant features

In [3]:
cls = add_pct(pd.read_csv("../../notebooks/_elena/data/bcn_noise_class_ml_dataset.csv").dropna())
reg = add_pct(pd.read_csv("../../notebooks/_elena/data/bcn_noise_regre_ml_dataset.csv").dropna())
print('class:', cls.shape, '| regre:', reg.shape)
cls[PCT_COLS].describe().loc[["min", "mean", "max"]].round(3)

class: (12854, 32) | regre: (12854, 32)


,dist_to_trunk_pct,dist_to_primary_pct,dist_to_secondary_pct,dist_to_tertiary_pct,dist_to_residential_pct,dist_to_living_street_pct,betweenness_pct,closeness_global_pct,closeness_400_pct,straightness_pct
min,0.002,0.035,0.098,0.166,0.408,0.148,0.004,0.0,0.0,0.0
mean,0.500,0.500,0.500,0.500,0.500,0.500,0.500,0.5,0.5,0.5
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.0,1.0,1.0


## Train + save both variants

In [4]:
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)
yc = cls['noise_day'].to_numpy()
yr = reg['noise_day'].to_numpy(dtype=float)

def dump(obj, name):
    with open(f'{MODELS_DIR}/{name}', 'wb') as f:
        pickle.dump(obj, f)

def train_and_save(feat_cols, suffix):
    """Fit scaler + 3 classifiers + 3 regressors on feat_cols; save all; return held-out metrics."""
    scaler = StandardScaler().fit(cls[feat_cols])
    dump(scaler, f'Sscaler_{suffix}.pkl')
    dump(list(feat_cols), f'feature_columns_{suffix}.pkl')

    # classification
    Xc = scaler.transform(cls[feat_cols])
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=42)
    rows = []
    for name, model, fname in [
        ('Logistic Regression', LogisticRegression(max_iter=2000), f'logreg_class_{suffix}.pkl'),
        ('XGBoost', xgb.XGBClassifier(), f'xgb_class_{suffix}.pkl'),
        ('Random Forest', RandomForestClassifier(random_state=42), f'rf_class_{suffix}.pkl'),
    ]:
        model.fit(Xc_tr, yc_tr); yp = model.predict(Xc_te); dump(model, fname)
        rows.append({'variant': suffix, 'task': 'class', 'model': name,
                     'accuracy': accuracy_score(yc_te, yp),
                     'macro_f1': f1_score(yc_te, yp, average='macro'),
                     'within_1': np.mean(np.abs(yp - yc_te) <= 1)})

    # regression
    Xr = scaler.transform(reg[feat_cols])
    Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=42)
    for name, model, fname in [
        ('Linear Regression', LinearRegression(), f'linreg_regre_{suffix}.pkl'),
        ('XGBoost', xgb.XGBRegressor(), f'xgb_regre_{suffix}.pkl'),
        ('Random Forest', RandomForestRegressor(random_state=42), f'rf_regre_{suffix}.pkl'),
    ]:
        model.fit(Xr_tr, yr_tr); yp = model.predict(Xr_te); dump(model, fname)
        rows.append({'variant': suffix, 'task': 'regre', 'model': name,
                     'r2': r2_score(yr_te, yp),
                     'mae': mean_absolute_error(yr_te, yp),
                     'rmse': np.sqrt(mean_squared_error(yr_te, yp))})
    return pd.DataFrame(rows)

metrics = pd.concat([train_and_save(SI10_FEATS, 'si'),
                     train_and_save(SI18_FEATS, 'si18')], ignore_index=True)
print('saved SI-10 (suffix _si) and SI-18 (suffix _si18) models')

saved SI-10 (suffix _si) and SI-18 (suffix _si18) models


## Barcelona held-out — classification

In [5]:
metrics[metrics.task == 'class'].pivot(index='model', columns='variant',
        values='accuracy').rename(columns={'si': 'SI-10', 'si18': 'SI-18'}).round(4)

variant,SI-10,SI-18
model,,
Logistic Regression,0.5994,0.6266
Random Forest,0.7480,0.7460
XGBoost,0.7417,0.7464


## Barcelona held-out — regression (R²)

In [6]:
metrics[metrics.task == 'regre'].pivot(index='model', columns='variant',
        values='r2').rename(columns={'si': 'SI-10', 'si18': 'SI-18'}).round(4)

variant,SI-10,SI-18
model,,
Linear Regression,0.4159,0.4804
Random Forest,0.6943,0.7032
XGBoost,0.6787,0.6820


## Sanity

Percentile rank is monotonic, so within Barcelona:
- **SI-18** RF/XGBoost ≈ the original **full-18** model (RF 0.749 acc / 0.704 R², XGB 0.744 / 0.681);
- **SI-10** RF/XGBoost ≈ the ablation's raw **only-distances+centrality** (RF 0.751 / 0.696, XGB 0.742 / 0.679).

If these match, the transforms preserved all the Barcelona signal and any change on the other cities is
pure transfer effect. The linear models may improve slightly (percentile ranks linearise skewed distances).